# Traveling Salesman Problem: ADAPT-VQE vs Momentum Builders

This notebook benchmarks four approaches on the Traveling Salesman Problem problem set used by the local `AnsatzBenchmarking` modules:

1. **`MomentumBuilder`**: momentum-based ansatz growth followed by VQE refinement.
2. **`MomentumBuilder+SA (Phased)`**: the two-stage Monte Carlo pipeline from `momentum_sa_phased`.
3. **`MomentumBuilder+SA (Merged)`**: the interleaved momentum/Monte Carlo pipeline from `momentum_sa_merged`.
4. **`ADAPT-VQE`**: adaptive ansatz construction using the shared implementation in `AdaptVQE.py`.

The structure mirrors the existing performance-evaluation notebooks while comparing both Monte Carlo variants against the plain momentum builder and ADAPT-VQE.


In [ ]:
%matplotlib inline
import os
import sys
import time
import pathlib
import numpy as np
import pandas as pd

cwd = pathlib.Path.cwd()
ansatz_pruning_dir = cwd / "AnsatzPruning" if (cwd / "AnsatzPruning").exists() else cwd
repo_root = ansatz_pruning_dir.parent

os.environ.setdefault("MPLCONFIGDIR", str(ansatz_pruning_dir / ".mplconfig"))
os.environ.setdefault("XDG_CACHE_HOME", str(ansatz_pruning_dir / ".cache"))
pathlib.Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
pathlib.Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import matplotlib
import matplotlib.pyplot as plt

try:
    import seaborn as sns
except ModuleNotFoundError:
    sns = None

for p in [repo_root, ansatz_pruning_dir]:
    p_str = str(p)
    if p_str not in sys.path:
        sys.path.insert(0, p_str)

from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Statevector
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA

import MomentumBuilder as RawMomentumBuilder
import MomentumMonteCarlo as RawMomentumMonteCarlo
from AdaptVQE import (
    build_reference_state,
    build_operator_pool,
    run_adapt_vqe,
    extract_result_metrics,
)

if sns is not None:
    sns.set_style("whitegrid")
else:
    plt.style.use("default")
plt.rcParams['figure.figsize'] = (12, 8)

from AnsatzBenchmarking.Problems.tsp.TSPProblems import TSPProblemSet

In [ ]:
def wrap_cost(cost):
    """Ensure cost_func-style outputs become Python scalars."""
    if isinstance(cost, np.ndarray):
        return float(cost.item() if cost.size == 1 else cost[0])
    if isinstance(cost, list):
        return float(cost[0] if len(cost) > 0 else 0.0)
    return float(cost)



def make_rx_reference_components(num_qubits: int):
    circuit = QuantumCircuit(num_qubits)
    ansatz = QuantumCircuit(num_qubits)
    for i in range(num_qubits):
        ansatz.rx(Parameter(f"angle{i}"), i)
    return circuit, ansatz, [1.0] * num_qubits, list(range(num_qubits))



def circuit_energy(circuit, hamiltonian):
    state = Statevector.from_instruction(circuit)
    return float(np.real(state.expectation_value(hamiltonian)))



def _finalize_builder_state(builder, circuit, optimized_params):
    builder.circuit = circuit
    builder.optimized_params = np.array(optimized_params, dtype=float).tolist()

    ordered = list(builder.circuit.parameters)
    if len(ordered) == len(builder.optimized_params):
        builder.optimized_param_dict = {
            p.name: float(v) for p, v in zip(ordered, builder.optimized_params)
        }
        builder.initial_point = np.array(builder.optimized_params, dtype=float)
    return builder.circuit



class NotebookMomentumBuilder:
    label = "MomentumBuilder"

    def __init__(self, hamiltonian):
        self.hamiltonian = hamiltonian
        self.circuit = None
        self.optimized_params = None
        self.optimized_param_dict = None
        self.initial_point = None

    def getCircuit(self):
        if self.circuit is not None:
            return self.circuit

        H = self.hamiltonian
        n_qubits = H.num_qubits
        circuit, ansatz, params, inds = make_rx_reference_components(n_qubits)

        if n_qubits < 2:
            built_circuit = circuit.compose(ansatz)
            return _finalize_builder_state(self, built_circuit, params)

        observables = [*H.paulis, H]
        built_circuit, built_params = RawMomentumBuilder.MomentumBuilder(
            params=params,
            inds=inds,
            ansatz=ansatz,
            circuit=circuit,
            hamiltonian=observables,
            estimator=StatevectorEstimator(),
            beta1=0.9,
            beta2=0.99,
            iters=MOMENTUM_ITERS,
            return_ansatz_and_params=True,
        )
        return _finalize_builder_state(self, built_circuit, built_params)



class _NotebookMomentumSABase:
    mc_builder = None
    label = None

    def __init__(self, hamiltonian):
        self.hamiltonian = hamiltonian
        self.circuit = None
        self.optimized_params = None
        self.optimized_param_dict = None
        self.initial_point = None

    def getCircuit(self):
        if self.circuit is not None:
            return self.circuit

        H = self.hamiltonian
        n_qubits = H.num_qubits
        circuit, ansatz, params, inds = make_rx_reference_components(n_qubits)

        if n_qubits < 2:
            built_circuit = circuit.compose(ansatz)
            return _finalize_builder_state(self, built_circuit, params)

        built_circuit, built_params, _ = self.mc_builder(
            params=params,
            inds=inds,
            ansatz=ansatz,
            circuit=circuit,
            hamiltonian=H,
            estimator=StatevectorEstimator(),
            beta1=0.9,
            beta2=0.99,
            iters=MOMENTUM_ITERS,
            optimization_runs=MONTE_CARLO_RUNS,
        )
        return _finalize_builder_state(self, built_circuit, built_params)



class NotebookMomentumSAPhasedBuilder(_NotebookMomentumSABase):
    label = "MomentumBuilder+SA (Phased)"
    mc_builder = staticmethod(RawMomentumMonteCarlo.momentum_sa_phased)



class NotebookMomentumSAMergedBuilder(_NotebookMomentumSABase):
    label = "MomentumBuilder+SA (Merged)"
    mc_builder = staticmethod(RawMomentumMonteCarlo.momentum_sa_merged)



def _make_x0_from_builder(builder, circuit):
    ordered_names = [p.name for p in circuit.parameters]

    if getattr(builder, "optimized_param_dict", None):
        return np.array([
            float(builder.optimized_param_dict.get(name, 0.0))
            for name in ordered_names
        ], dtype=float)

    if getattr(builder, "optimized_params", None) is not None:
        cand = np.array(builder.optimized_params, dtype=float)
        if len(cand) == len(ordered_names):
            return cand

    return None



def evaluate_vqe_builder(
    builder_class,
    problem_set,
    trials,
    objective_transform,
    exact_transform,
    success_tolerance,
    optimizer_maxiter,
):
    all_results = []
    problems = problem_set.getProblemSet()

    for trial in range(trials):
        estimator = StatevectorEstimator()

        for i, (h, exact) in enumerate(problems):
            objective = objective_transform(h)
            exact_energy = exact_transform(exact)
            builder = builder_class(objective)

            build_start = time.time()
            circuit = builder.getCircuit()
            build_time = time.time() - build_start

            x0 = _make_x0_from_builder(builder, circuit)
            vqe_optimizer = COBYLA(maxiter=max(optimizer_maxiter, len(circuit.parameters) + 2))
            vqe = VQE(estimator=estimator, ansatz=circuit, optimizer=vqe_optimizer)
            if x0 is not None:
                vqe.initial_point = x0

            opt_start = time.time()
            vqe_result = vqe.compute_minimum_eigenvalue(objective)
            opt_time = time.time() - opt_start

            energy = float(np.real(vqe_result.eigenvalue))
            error = abs(energy - exact_energy) if exact_energy is not None else np.nan
            optimizer_result = getattr(vqe_result, 'optimizer_result', None)
            converged = getattr(optimizer_result, 'success', None)
            if converged is None:
                converged = bool(error <= success_tolerance)

            all_results.append({
                'ansatz_type': builder_class.label,
                'problem_index': i + 1,
                'trial': trial + 1,
                'energy': energy,
                'exact_energy': exact_energy,
                'error': error,
                'time': build_time + opt_time,
                'build_time': build_time,
                'opt_time': opt_time,
                'params': len(circuit.parameters),
                'depth': circuit.depth(),
                'converged': converged,
                'adapt_iterations': np.nan,
                'adapt_initial_energy': np.nan,
                'adapt_energy_drop': np.nan,
                'adapt_history_length': np.nan,
                'termination_criterion': None,
            })

    return all_results



def evaluate_adapt_vqe(
    problem_set,
    trials,
    objective_transform,
    exact_transform,
    success_tolerance,
    adapt_iteration_budget,
    adapt_optimizer_maxiter,
):
    all_results = []
    problems = problem_set.getProblemSet()

    for trial in range(trials):
        for i, (h, exact) in enumerate(problems):
            objective = objective_transform(h)
            exact_energy = exact_transform(exact)
            initial_state = build_reference_state(objective.num_qubits, angle=1.0)
            initial_energy = circuit_energy(initial_state, objective)

            start = time.time()
            result = run_adapt_vqe(
                hamiltonian=objective,
                initial_state=initial_state,
                operator_pool=build_operator_pool(objective.num_qubits),
                max_iterations=adapt_iteration_budget(objective),
                gradient_threshold=1e-6,
                eigenvalue_threshold=1e-8,
                optimizer_maxiter=adapt_optimizer_maxiter,
            )
            elapsed = time.time() - start

            metrics = extract_result_metrics(result)
            history = metrics['eigenvalue_history'] or []
            energy = float(metrics['eigenvalue'])
            error = abs(energy - exact_energy) if exact_energy is not None else np.nan
            converged = bool(error <= success_tolerance)

            all_results.append({
                'ansatz_type': 'ADAPT-VQE',
                'problem_index': i + 1,
                'trial': trial + 1,
                'energy': energy,
                'exact_energy': exact_energy,
                'error': error,
                'time': elapsed,
                'build_time': elapsed,
                'opt_time': 0.0,
                'params': int(metrics['num_parameters']),
                'depth': metrics['ansatz'].depth() if metrics['ansatz'] is not None else np.nan,
                'converged': converged,
                'adapt_iterations': metrics['num_iterations'],
                'adapt_initial_energy': initial_energy,
                'adapt_energy_drop': initial_energy - energy,
                'adapt_history_length': len(history),
                'termination_criterion': metrics['termination_criterion'],
            })

    return all_results


In [ ]:
# Load problem set and configure benchmark settings
print("Loading Traveling Salesman Problem problem set...")
problem_set = TSPProblemSet()
problems = problem_set.getProblemSet()
print(f"Loaded {len(problems)} Traveling Salesman Problem problems\n")

TRIALS = 10
MOMENTUM_ITERS = 3
MONTE_CARLO_RUNS = 200
BUILDER_OPTIMIZER_MAXITER = 100
ADAPT_OPTIMIZER_MAXITER = 200
SUCCESS_TOLERANCE = 0.5
RESULTS_CSV = "AnsatzComparisonTSPAdapt.csv"
MAIN_PNG = "ansatz_comparison_tsp_adapt.png"
DETAIL_PNG = "ansatz_detailed_comparison_tsp_adapt.png"
problem_label = "Traveling Salesman Problem"

METHOD_ORDER = [
    "MomentumBuilder",
    "MomentumBuilder+SA (Phased)",
    "MomentumBuilder+SA (Merged)",
    "ADAPT-VQE",
]
METHOD_COLORS = {
    "MomentumBuilder": "#2ecc71",
    "MomentumBuilder+SA (Phased)": "#3498db",
    "MomentumBuilder+SA (Merged)": "#9b59b6",
    "ADAPT-VQE": "#f39c12",
}

objective_transform = lambda h: -h
exact_transform = lambda exact: -float(exact)
adapt_iteration_budget = lambda h: max(12, h.num_qubits)

print("Running benchmarks...")
print("=" * 60)


In [ ]:
# Evaluate MomentumBuilder
print("\n" + "=" * 60)
print("MomentumBuilder:")
print("=" * 60)
momentum_results = evaluate_vqe_builder(
    NotebookMomentumBuilder,
    problem_set,
    trials=TRIALS,
    objective_transform=objective_transform,
    exact_transform=exact_transform,
    success_tolerance=SUCCESS_TOLERANCE,
    optimizer_maxiter=BUILDER_OPTIMIZER_MAXITER,
)
print(f"Collected {len(momentum_results)} MomentumBuilder results")


In [ ]:
# Evaluate MomentumBuilder + SA (Phased)
print("\n" + "=" * 60)
print("MomentumBuilder + SA (Phased):")
print("=" * 60)
phased_results = evaluate_vqe_builder(
    NotebookMomentumSAPhasedBuilder,
    problem_set,
    trials=TRIALS,
    objective_transform=objective_transform,
    exact_transform=exact_transform,
    success_tolerance=SUCCESS_TOLERANCE,
    optimizer_maxiter=BUILDER_OPTIMIZER_MAXITER,
)
print(f"Collected {len(phased_results)} MomentumBuilder+SA (Phased) results")


In [ ]:
# Evaluate MomentumBuilder + SA (Merged)
print("\n" + "=" * 60)
print("MomentumBuilder + SA (Merged):")
print("=" * 60)
merged_results = evaluate_vqe_builder(
    NotebookMomentumSAMergedBuilder,
    problem_set,
    trials=TRIALS,
    objective_transform=objective_transform,
    exact_transform=exact_transform,
    success_tolerance=SUCCESS_TOLERANCE,
    optimizer_maxiter=BUILDER_OPTIMIZER_MAXITER,
)
print(f"Collected {len(merged_results)} MomentumBuilder+SA (Merged) results")


In [ ]:
# Evaluate ADAPT-VQE
print("\n" + "=" * 60)
print("ADAPT-VQE:")
print("=" * 60)
adapt_results = evaluate_adapt_vqe(
    problem_set,
    trials=TRIALS,
    objective_transform=objective_transform,
    exact_transform=exact_transform,
    success_tolerance=SUCCESS_TOLERANCE,
    adapt_iteration_budget=adapt_iteration_budget,
    adapt_optimizer_maxiter=ADAPT_OPTIMIZER_MAXITER,
)
print(f"Collected {len(adapt_results)} ADAPT-VQE results")

all_results = momentum_results + phased_results + merged_results + adapt_results
df = pd.DataFrame(all_results)
if len(df) > 0:
    df['ansatz_type'] = pd.Categorical(df['ansatz_type'], categories=METHOD_ORDER, ordered=True)
    df = df.sort_values(['ansatz_type', 'problem_index', 'trial']).reset_index(drop=True)
print(f"\n{'=' * 60}")
print(f"Completed: {len(df)} successful trials")


In [ ]:
# Display summary statistics
if len(df) > 0:
    print("\n" + "=" * 60)
    print("SUMMARY STATISTICS")
    print("=" * 60)

    summary = df.groupby('ansatz_type', observed=False).agg({
        'energy': ['mean', 'std', 'min', 'max'],
        'error': ['mean', 'std', 'min', 'max'],
        'time': ['mean', 'std', 'min', 'max'],
        'params': ['mean', 'std'],
        'depth': ['mean', 'std'],
        'converged': 'mean',
    }).reindex(METHOD_ORDER).dropna(how='all')
    print(summary)

    adapt_only = df[df['ansatz_type'] == 'ADAPT-VQE']
    if len(adapt_only) > 0:
        print("\nADAPT-VQE diagnostics:")
        print(adapt_only[['adapt_iterations', 'adapt_initial_energy', 'adapt_energy_drop', 'adapt_history_length']].agg(['mean', 'std', 'min', 'max']))
        print("Termination criteria:", sorted(adapt_only['termination_criterion'].dropna().unique()))

    df.to_csv(RESULTS_CSV, index=False)
    print(f"\nResults saved to {RESULTS_CSV}")
else:
    print("No successful trials to display.")


In [ ]:
# Visualization 1: Main comparison plots
if len(df) > 0 and 'ansatz_type' in df.columns:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    order = METHOD_ORDER

    ax1 = axes[0, 0]
    df.boxplot(column='energy', by='ansatz_type', ax=ax1)
    ax1.set_title(f'{problem_label}: Final Energy Comparison', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Ansatz Type')
    ax1.set_ylabel('Energy')
    ax1.grid(True, alpha=0.3)
    plt.suptitle('')

    ax2 = axes[0, 1]
    df.boxplot(column='time', by='ansatz_type', ax=ax2)
    ax2.set_title(f'{problem_label}: Runtime Comparison', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Ansatz Type')
    ax2.set_ylabel('Time (seconds)')
    ax2.grid(True, alpha=0.3)
    plt.suptitle('')

    ax3 = axes[1, 0]
    param_means = df.groupby('ansatz_type', observed=False)['params'].mean().reindex(order).dropna()
    param_stds = df.groupby('ansatz_type', observed=False)['params'].std().reindex(param_means.index).fillna(0.0)
    x_pos = np.arange(len(param_means))
    ax3.bar(
        x_pos,
        param_means.values,
        yerr=param_stds.values,
        color=[METHOD_COLORS[label] for label in param_means.index],
        capsize=5,
    )
    ax3.set_title('Number of Parameters', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Ansatz Type')
    ax3.set_ylabel('Parameters')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(param_means.index, rotation=15, ha='right')
    ax3.grid(True, alpha=0.3, axis='y')

    ax4 = axes[1, 1]
    for ansatz_type in order:
        subset = df[df['ansatz_type'] == ansatz_type]
        if len(subset) > 0:
            ax4.scatter(
                subset['time'],
                subset['energy'],
                label=ansatz_type,
                alpha=0.7,
                s=60,
                color=METHOD_COLORS[ansatz_type],
            )
    ax4.set_title('Energy vs Time Trade-off', fontsize=14, fontweight='bold')
    ax4.set_xlabel('Time (seconds)')
    ax4.set_ylabel('Energy')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(MAIN_PNG, dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# Visualization 2: Detailed comparison plots
if len(df) > 0 and 'ansatz_type' in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    order = METHOD_ORDER

    ax1 = axes[0]
    for ansatz_type in order:
        subset = df[df['ansatz_type'] == ansatz_type]
        if len(subset) > 0:
            ax1.hist(
                subset['energy'],
                alpha=0.55,
                label=ansatz_type,
                bins=10,
                color=METHOD_COLORS[ansatz_type],
            )
    ax1.set_xlabel('Energy')
    ax1.set_ylabel('Frequency')
    ax1.set_title('Energy Distribution', fontsize=12, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2 = axes[1]
    for ansatz_type in order:
        subset = df[df['ansatz_type'] == ansatz_type]
        if len(subset) > 0:
            ax2.hist(
                subset['time'],
                alpha=0.55,
                label=ansatz_type,
                bins=10,
                color=METHOD_COLORS[ansatz_type],
            )
    ax2.set_xlabel('Time (seconds)')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Time Distribution', fontsize=12, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3 = axes[2]
    success_rate = df.groupby('ansatz_type', observed=False)['converged'].agg(['mean', 'count']).reindex(order).dropna()
    x_pos = np.arange(len(success_rate))
    ax3.bar(
        x_pos,
        success_rate['mean'].values * 100.0,
        color=[METHOD_COLORS[label] for label in success_rate.index],
    )
    ax3.set_ylabel('Success Rate (%)')
    ax3.set_title('Convergence Success Rate', fontsize=12, fontweight='bold')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(success_rate.index, rotation=15, ha='right')
    ax3.grid(True, alpha=0.3, axis='y')
    ax3.set_ylim([0, 105])

    plt.tight_layout()
    plt.savefig(DETAIL_PNG, dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# Visualization 3: Aggregate metric comparison
if len(df) > 0 and 'ansatz_type' in df.columns:
    fig, ax = plt.subplots(figsize=(12, 6))

    metric_names = ['Mean Energy', 'Mean Time', 'Mean Params']
    grouped = df.groupby('ansatz_type', observed=False).agg({
        'energy': 'mean',
        'time': 'mean',
        'params': 'mean',
    }).reindex(METHOD_ORDER).dropna(how='all')

    x = np.arange(len(metric_names))
    labels = list(grouped.index)
    width = min(0.8 / max(len(labels), 1), 0.3)

    for idx, label in enumerate(labels):
        values = [grouped.loc[label, 'energy'], grouped.loc[label, 'time'], grouped.loc[label, 'params']]
        offset = (idx - (len(labels) - 1) / 2) * width
        bars = ax.bar(x + offset, values, width, label=label, color=METHOD_COLORS[label], alpha=0.85)
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, height, f'{height:.3f}', ha='center', va='bottom', fontsize=8)

    ax.set_ylabel('Value', fontsize=12)
    ax.set_title(f'{problem_label}: Aggregate Metrics Comparison', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(metric_names)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()
